### SoRL Playground

This notebook demonstrates the SoRL post-training pipeline. Porting a OSS model, and adopt SoRL trainer to post-train the model accordingly. 

In [1]:
# Setup
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output
import time

# Disable MPS for stability
if hasattr(torch.backends, 'mps'):
    torch.backends.mps.is_available = lambda: False

from transformers import TrainingArguments, AutoTokenizer
from sorl.sorl_wrapper import SorlModelWrapper
from sorl.sorl_trainer import SorlTrainer

device = torch.device("cpu")
print(f"Using device: {device}")

Using device: cpu


In [2]:
# Initialize SoRL model
from sorl.sorl_wrapper import SorlModelWrapper
model_name = "Qwen/Qwen2.5-0.5B"
model = SorlModelWrapper.from_pretrained(
    model_name,
    abstract_vocab_size_list=[128],
)
model = model.to(device)
tokenizer = AutoTokenizer.from_pretrained(model_name) 

Some weights of Qwen2ForCausalLM were not initialized from the model checkpoint at Qwen/Qwen2.5-0.5B and are newly initialized: ['lm_head.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
The new lm_head weights will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


In [12]:
# 1. Attention_Mask blocks attention to pad token, label=-100 exempts loss computation for Query tokens
# 2. Therefore, for inserting abstract tokens, I should use 'attention_mask' to guide me, for loss computation, I rely on 'labels'
# 3. The issue is masked labels require an extra term that records position of query for each data point. 

In [7]:
# ============================================================
# SoRL Training with standalone Trainer
# ============================================================
from sorl.trainer import SoRLTrainer, SoRLConfig
from data.pt_dataset import get_dataset, evaluate_accuracy

config = SoRLConfig(
    # SoRL search
    num_rollouts=4, K=4, max_iterations=2,
    memory_span_abs=1792, memory_span_traj=1792, temperature=1.0,
    # Loss weights
    alpha_info_gain=10.0, alpha_abs=0.1, alpha_soft_zipf=1.0,
    # Optimizer
    lr=1e-5, weight_decay=0.01, warmup_steps=50, max_grad_norm=1.0,
    # Training
    batch_size=2, num_epochs=1,
    # Logging / Eval / Checkpoint
    log_every=10, eval_every=500, save_every=500, eval_samples=50,
    output_dir="./ckpt/sorl",
)

# Datasets
train_ds = get_dataset("gsm8k", split="train", tokenizer=tokenizer, max_length=256)
val_ds = get_dataset("gsm8k", split="test", tokenizer=tokenizer, max_length=256)

trainer = SoRLTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    val_dataset=val_ds,  # set val_ds here for periodic eval
    compute_accuracy=evaluate_accuracy,
    config=config,
    ddp=False,  # set True when using torchrun
    device=device
)

# history = trainer.train()

In [ ]:
history = trainer.train()

Total steps: 3737 | Steps/epoch: 3737 | Effective batch: 2
epoch 0.003/1 | remain: 8h39m50s | loss=-22.2708 base=23.6663 info=-5.3442 abs=20.7762 zipf=5.4270 | 


In [5]:
# ----- evaluation & logs ----- 
# (1). Accuracy
# (2). Stat on abstract sequences (Inner monologue)
self = trainer
self.raw_model.eval()
result = self.compute_accuracy(
    self.raw_model, self.tokenizer, self.val_dataset,
    self.device, 5,
)

In [ ]:
# ---- Inspect accuracy evaluation process ----
from data.pt_dataset import get_dataset, _filter_traj_tokens

val_ds = get_dataset("gsm8k", split="test", tokenizer=tokenizer, max_length=256)
item = val_ds[0]
input_ids = item["input_ids"].unsqueeze(0).to(device)
attention_mask = item["attention_mask"].unsqueeze(0).to(device)
prompt_len = item["prompt_len"]
base_vocab_size = model.vocab_sizes[0].item()
extract_fn = val_ds.extract_answer
max_new_tokens = 50

print(f"prompt_len: {prompt_len}")
print(f"total non-pad tokens: {attention_mask.sum().item()}")
print(f"answer tokens: {attention_mask.sum().item() - prompt_len}")

# Decode question prefix
question_text = tokenizer.decode(input_ids[0, :prompt_len], skip_special_tokens=True)
print(f"\n--- Question ---\n{question_text}")

# Decode reference full text (filter abstract tokens just in case)
ref_ids = input_ids[0][input_ids[0] < base_vocab_size]
ref_text = tokenizer.decode(ref_ids, skip_special_tokens=True)
gold_answer = extract_fn(ref_text)
print(f"\n--- Reference ---\n{ref_text}")
print(f"Gold answer: {gold_answer}")

# Generate from question prefix only
model.eval()
generated = model.generate(
    input_ids=input_ids[:, :prompt_len],
    max_new_tokens=max_new_tokens,
    temperature=0.0,
    K=4,
)
print(f"\n--- Generation ---")
print(f"Generated shape: {generated.shape} (prompt={prompt_len} + new={generated.shape[1]-prompt_len})")

# Filter abstract tokens and decode
traj_tokens = _filter_traj_tokens(generated, base_vocab_size)
full_text = tokenizer.decode(traj_tokens[0], skip_special_tokens=True)
pred_answer = extract_fn(full_text)
print(f"Full text:\n{full_text}")
print(f"Pred answer: {pred_answer}")

# Compare
print(f"\n--- Result ---")
print(f"Gold: {gold_answer} | Pred: {pred_answer} | Match: {pred_answer == gold_answer if pred_answer and gold_answer else 'N/A'}")